# 📡 Brand Sentiment & Image Index
**Track Reddit + Twitter brand perception → compute a 0–100 Image Index → link to brand equity**

Run cells top to bottom. The Streamlit app opens via an `ngrok` public URL at the end.

In [ ]:
# ── CELL 1: Install dependencies ─────────────────────────────────────────────
!pip install -q streamlit pyngrok vaderSentiment textblob praw wordcloud plotly snscrape
!python -m textblob.download_corpora

In [ ]:
# ── CELL 2: Upload project files ──────────────────────────────────────────────
# Option A — paste your GitHub raw URLs here:
import urllib.request, os

BASE_URL = ""  # e.g. "https://raw.githubusercontent.com/yourname/brand-sentiment/main/"
FILES    = ["app.py", "scraper.py", "sentiment.py", "image_index.py"]

if BASE_URL:
    for f in FILES:
        urllib.request.urlretrieve(BASE_URL + f, f)
    print("✅ Files downloaded from GitHub")
else:
    # Option B — upload manually
    from google.colab import files
    print("Upload app.py, scraper.py, sentiment.py, image_index.py")
    uploaded = files.upload()
    print("✅ Uploaded:", list(uploaded.keys()))

In [ ]:
# ── CELL 3: (Optional) Set Reddit API credentials ────────────────────────────
# Skip this cell if you want to use demo data (no credentials needed)
#
# To get credentials:
#   1. Go to https://www.reddit.com/prefs/apps
#   2. Create a 'script' app
#   3. Copy client_id and client_secret below

import os
os.environ["REDDIT_CLIENT_ID"]     = ""   # ← paste here
os.environ["REDDIT_CLIENT_SECRET"] = ""   # ← paste here

print("Reddit credentials set" if os.environ.get("REDDIT_CLIENT_ID") else "Demo mode — no credentials set")

In [ ]:
# ── CELL 4: Start Streamlit via ngrok ────────────────────────────────────────
from pyngrok import ngrok, conf
import subprocess, time, threading

# Optional: set your ngrok auth token for persistent tunnels
# conf.get_default().auth_token = "your_ngrok_token_here"

# Kill any existing Streamlit processes
!pkill -f streamlit 2>/dev/null || true
time.sleep(1)

# Start Streamlit in background
process = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--browser.gatherUsageStats", "false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

time.sleep(4)  # wait for Streamlit to boot

# Open ngrok tunnel
public_url = ngrok.connect(8501)
print("\n" + "="*60)
print(f"  🚀 App is LIVE at: {public_url}")
print("="*60)
print("  Click the link above to open the dashboard.")
print("  Keep this cell running. Stop it to shut down.")

In [ ]:
# ── CELL 5: (Optional) Quick local test — run analysis without UI ─────────────
from scraper     import collect_mentions
from sentiment   import analyze
from image_index import compute_all_indices

brands = ["Apple", "Tesla", "Nike"]

df_raw      = collect_mentions(brands, reddit_limit=30, twitter_limit=30)
df_analyzed = analyze(df_raw)
summary     = compute_all_indices(df_analyzed, brands)

print("\n📊 Brand Image Index Summary")
print(summary[["Brand", "Image Index", "Sentiment Score", "Total Mentions", "Trend"]].to_string(index=False))